# 01 · OULU-NPU Protocol II dataset audit

**QPAD — Calibration-Set Composition Bias in INT8 Face Presentation Attack Detection**
CS 228 Biometric Security with AI · San José State University · Ananya Tyagi & Vindhya Channarayapatna Ravishankar

**Priority:** 0 (Checkpoint 1)

**Purpose.** Create one manifest row per video and verify counts, Protocol II membership, subject leakage, duplicates, annotations, frame coverage and full decodability.

**In plain words.** We check the dataset is exactly what the protocol promises *before* training anything. Each video is copied alone into temporary storage, inspected with FFmpeg, and deleted. Progress is saved every few videos, so a Colab disconnect only loses a few minutes.

**Expected inputs**
- Archives and the header indexes from notebook 00.

**Expected outputs**
- `QPAD_Project/manifests/oulu_manifest.csv` and `.parquet`
- `QPAD_Project/reports/dataset_audit.json` / `.md`
- `QPAD_Project/logs/nb01.log`
- `QPAD_Project/manifests/audit_progress.csv` (resume file)

**Safety rules.** The OULU-NPU dataset is access-restricted. This notebook never uploads data, never creates
share links, never prints credentials, and never writes videos or face crops to the Git repository.
Long operations run only after you change `DRY_RUN = True` to `False` and read the pre-flight summary.

In [ ]:
# --- Step 1: mount Google Drive (private storage) -------------------------------------------
# Authentication happens in Colab's own pop-up; nothing about it is stored in this notebook.
import os, sys, json, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not os.environ.get("QPAD_SKIP_DRIVE_MOUNT"):
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Drive mount skipped (not in Colab, or test mode).")

In [ ]:
# --- Step 2: make the QPAD code importable ------------------------------------------------------
# Option A (default): upload the whole `qpad/` repository folder to
#   MyDrive/QPAD_Project/code/qpad
# Option B: set REPO_URL to your PRIVATE GitHub repository; Colab/GitHub will ask you to authenticate.
#   Never paste access tokens into this notebook.
REPO_URL = ""  # e.g. "https://github.com/<account>/qpad.git"
QPAD_CODE_DIR = Path(os.environ.get("QPAD_CODE_DIR", "/content/drive/MyDrive/QPAD_Project/code/qpad"))
if REPO_URL:
    QPAD_CODE_DIR = Path("/content/qpad")
    if not (QPAD_CODE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(QPAD_CODE_DIR)], check=True)
if not (QPAD_CODE_DIR / "src" / "qpad").is_dir():
    raise FileNotFoundError(f"QPAD code not found at {QPAD_CODE_DIR}. Upload the repo folder or set REPO_URL.")
sys.path.insert(0, str(QPAD_CODE_DIR / "src"))

if IN_COLAB and not os.environ.get("QPAD_SKIP_PIP"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(QPAD_CODE_DIR / "requirements-colab.txt")], check=True)
import qpad
print("qpad", qpad.__version__, "from", QPAD_CODE_DIR)

In [ ]:
# --- Step 3: authoritative paths -------------------------------------------------------------------
# These are the ONLY data/output locations. pathlib handles any spaces in paths.
from qpad.config import ProjectPaths, load_config, get_logger, set_global_seed, disk_free_gb

DATASET_ROOT = Path("/content/drive/MyDrive/QPAD_Project/OULU_NPU/raw")
TRAIN_TAR = DATASET_ROOT / "Train_files-002.tar"
DEV_TAR = DATASET_ROOT / "Dev_files-001.tar"
TEST_TAR = DATASET_ROOT / "Test_files-003.tar"
METADATA_ZIP = DATASET_ROOT / "OULU-NPU-20261010T032906Z-1-004.zip"
PROJECT_DRIVE_ROOT = Path("/content/drive/MyDrive/QPAD_Project")
LOCAL_WORK_ROOT = Path("/content/qpad_work")

cfgs = load_config(QPAD_CODE_DIR / "configs")
paths = ProjectPaths.from_config(cfgs["dataset"])
if not paths.overrides and not cfgs["_overrides"]:  # YAML must agree with the constants above
    assert paths.dataset_root == DATASET_ROOT and paths.project_drive_root == PROJECT_DRIVE_ROOT
    assert paths.archive("train") == TRAIN_TAR and paths.archive("dev") == DEV_TAR
    assert paths.archive("test") == TEST_TAR and paths.archive("metadata") == METADATA_ZIP
    assert paths.local_work_root == LOCAL_WORK_ROOT
for o in cfgs["_overrides"]:
    print("WARNING (test mode):", o)
paths.ensure_output_dirs()
print(paths.describe())

In [ ]:
# --- Step 4: software, GPU and disk information ------------------------------------------------
from qpad.reproducibility import environment_info, git_commit
from qpad.nbutils import df_h, tqdm_progress, preflight

ENV_INFO = environment_info()
print(json.dumps(ENV_INFO, indent=1, default=str))
print("git:", git_commit(QPAD_CODE_DIR))
print(df_h(LOCAL_WORK_ROOT if LOCAL_WORK_ROOT.exists() else Path("/content") if Path("/content").exists() else Path.cwd()))

In [ ]:
# --- Step 5: run configuration and seeds ---------------------------------------------------------
DRY_RUN = True                 # change to False ONLY after the dry run looks correct
DRY_RUN_VIDEOS_PER_CLASS = 2   # per split and class, deterministic (first IDs)
FORCE_REBUILD = False          # True re-creates outputs that already exist
RETRY_FAILED = False           # True re-processes items recorded as failed
RANDOM_SEED = 42

if os.environ.get("QPAD_FORCE_DRY_RUN"):
    DRY_RUN = True
SEED_INFO = set_global_seed(RANDOM_SEED)
logger = get_logger("qpad.nb01", paths.logs / "nb01.log")
logger.info("DRY_RUN=%s seed=%s", DRY_RUN, RANDOM_SEED)
from qpad.pipeline import output_paths
OUT = output_paths(paths, DRY_RUN)
OUT.ensure_output_dirs()
print("Outputs of this run go to:", OUT.project_drive_root)

## 1. Index-level checks (fast)
Names, labels, counts, leakage and annotation presence come from the archive indexes alone.

In [ ]:
from qpad.pipeline import build_video_manifest, dry_run_subset
from qpad.manifest import validate_structure, checks_to_markdown
from IPython.display import Markdown
VIDEOS, PROTOCOL_LISTS = build_video_manifest(paths, cfgs["dataset"])
STRUCT = validate_structure(VIDEOS, cfgs["dataset"], PROTOCOL_LISTS or None)
display(Markdown(checks_to_markdown(STRUCT)))
print("video members:", len(VIDEOS), "| Protocol II:", int(VIDEOS["in_protocol"].sum()))
display(VIDEOS[VIDEOS.in_protocol].groupby(["split", "class"]).size().unstack())

## 2. Pre-flight
Read this before a full run (~2,970 videos).

In [ ]:
from qpad.audit import estimate_audit_hours
TARGET = dry_run_subset(VIDEOS, DRY_RUN_VIDEOS_PER_CLASS) if DRY_RUN else VIDEOS[VIDEOS.in_protocol]
acfg = cfgs["dataset"]["audit"]
assert len(TARGET) > 0, "nothing to audit"
assert TARGET["parse_error"].eq("").all(), "fix filename parse errors first"
preflight("per-video audit (extract one video -> ffprobe -> ffmpeg decode -> delete)", len(TARGET),
          [paths.archive(k) for k in ("train", "dev", "test")],
          [OUT.manifests / "oulu_manifest.csv", OUT.reports / "dataset_audit.md"], LOCAL_WORK_ROOT, acfg,
          RANDOM_SEED, DRY_RUN,
          f"~{estimate_audit_hours(len(TARGET), TARGET.file_size_bytes.mean() / 1e6, workers=acfg['workers'])} h "
          "(assumes ~60 MB/s Drive reads)")

## 3. Run the audit (resumable)
Re-running this cell skips videos already recorded in `audit_progress.csv`. Failed videos are kept and reported; set `RETRY_FAILED = True` to retry them.

In [ ]:
from qpad.pipeline import run_dataset_audit
RESULT = run_dataset_audit(paths, cfgs["dataset"], DRY_RUN, DRY_RUN_VIDEOS_PER_CLASS, RETRY_FAILED, logger,
                           tqdm_progress("audit"))
print("OVERALL STATUS:", RESULT["status"])

In [ ]:
display(Markdown(checks_to_markdown(RESULT["checks"])))
M = RESULT["manifest"]
print("failed videos:", int((M["status"] == "failed").sum()))
display(M[M["status"] == "failed"][["video_id", "error"]].head(20))
print("resolutions:", RESULT["summary"]["resolution_counts"])
print("frame counts:", RESULT["summary"]["frame_count"], "| duration (s):", RESULT["summary"]["duration_s"])
print("annotation frame bases:", M.loc[M.in_protocol, "ann_frame_base"].value_counts(dropna=False).to_dict())

In [ ]:
# --- Final summary -----------------------------------------------------------------------------
print("Status:", RESULT["status"], "(PASS requires every required check to run and pass)")
print("Manifest:", OUT.manifests / "oulu_manifest.csv")
print("Report  :", OUT.reports / "dataset_audit.md")
if DRY_RUN:
    print("DRY RUN: status INCOMPLETE is expected. Set DRY_RUN = False for the real audit.")

**Next notebook:** `02_preprocess_oulu.ipynb`